In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(
    "../data/archive/Friday-WorkingHours-Afternoon-DDoS.pcap_ISCX.csv"
)

print(df.shape)

(225745, 79)


In [5]:
print(df.isna().sum().sum())

4


## Observation:
 there are in total 4 missing values.

In [7]:
##Infinte Values(in FLow Bytes/s and Flow Packets/s as noticed in EDA)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(df.isna().sum().sum())

df.dropna(inplace=True)

print(df.shape)

68
(225711, 79)


## Observations:
 Initial data quality analysis revealed 4 missing values in the original dataset. Additionally, 30 and 34 infinite values were detected in Flow Bytes/s and Flow Packets/s, respectively. After converting infinite values to NaN, a total of 68 missing entries were identified. These invalid values were concentrated within 34 unique records, resulting in the removal of only 34 rows (approximately 0.015% of the dataset).

In [10]:
duplicates = df.duplicated().sum()
print(duplicates)

dup_mask = df.duplicated()

print(
    df[dup_mask][" Label"]
    .value_counts()
)

df.drop_duplicates(inplace=True)

print(df.shape)

2629
 Label
BENIGN    2618
DDoS        11
Name: count, dtype: int64
(223082, 79)


## Observation:
 Duplicate analysis revealed 2,629 duplicate records, of which 2,618 belonged to the BENIGN class and only 11 belonged to the DDoS class. This indicates that duplicate flow patterns are predominantly associated with normal network activity. To prevent redundancy and reduce potential bias, duplicate records were removed prior to further analysis.

In [ ]:
constant_cols = []

for col in df.columns:
    if df[col].nunique() == 1:
        constant_cols.append(col)

print(constant_cols)


[' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' CWE Flag Count', 'Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [12]:
constant_cols = [
    ' Bwd PSH Flags',
    ' Fwd URG Flags',
    ' Bwd URG Flags',
    ' CWE Flag Count',
    'Fwd Avg Bytes/Bulk',
    ' Fwd Avg Packets/Bulk',
    ' Fwd Avg Bulk Rate',
    ' Bwd Avg Bytes/Bulk',
    ' Bwd Avg Packets/Bulk',
    'Bwd Avg Bulk Rate'
]

df.drop(columns=constant_cols, inplace=True)

print(df.shape)

(223082, 69)


## Observation:
 Ten constant features were identified in the dataset. Since these features exhibited no variation across all records, they contained no discriminatory information and were removed during preprocessing. Eliminating constant features reduces dimensionality and prevents unnecessary computation during statistical and machine learning analysis.

In [ ]:
#Correlation Analysis:
X = df.drop(columns=[" Label"])
corr_matrix = X.corr(numeric_only=True)

corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr = [
    column
    for column in upper.columns
    if any(upper[column] > 0.95)
]

print("Highly correlated features:")
print(high_corr)
print("\nCount:", len(high_corr))



Highly correlated features:
[' Total Backward Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Std', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', ' Flow IAT Max', 'Fwd IAT Total', ' Fwd IAT Std', ' Fwd IAT Max', ' Bwd IAT Max', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Packet Length Std', ' SYN Flag Count', ' ECE Flag Count', ' Average Packet Size', ' Avg Fwd Segment Size', ' Avg Bwd Segment Size', ' Fwd Header Length.1', 'Subflow Fwd Packets', ' Subflow Fwd Bytes', ' Subflow Bwd Packets', ' Subflow Bwd Bytes', ' Active Min', ' Idle Max']

Count: 26


In [15]:
X_reduced = X.drop(columns=high_corr)

X_full = X.copy()
print(X_full.shape)
print(X_reduced.shape)

(223082, 68)
(223082, 42)


## Observation:
 Correlation analysis identified 26 highly correlated features (correlation coefficient > 0.95). These features likely contain redundant information and may increase dimensionality without contributing significant additional knowledge. A reduced feature set was therefore created by removing highly correlated features for subsequent experiments.

In [19]:
##Feature Scaling:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_reduced)
print(X_scaled.shape)


scaled_df = pd.DataFrame(
    X_scaled,
    columns=X_reduced.columns
)

print(scaled_df.mean().head())
print(scaled_df.std().head())

(223082, 42)
 Destination Port              1.141547e-16
 Flow Duration                 2.853867e-17
 Total Fwd Packets            -1.936552e-17
Total Length of Fwd Packets   -5.911581e-17
 Fwd Packet Length Max         2.038476e-17
dtype: float64
 Destination Port              1.000002
 Flow Duration                 1.000002
 Total Fwd Packets             1.000002
Total Length of Fwd Packets    1.000002
 Fwd Packet Length Max         1.000002
dtype: float64


## Observation:
 Standardization was applied using StandardScaler to ensure that all features contribute equally during anomaly detection. Verification confirmed that the transformed features have approximately zero mean and unit variance, indicating successful scaling.
